# Intro to RAG pipelines

In [5]:
!uv add langchain-ollama

Resolved 75 packages in 1ms
Checked 68 packages in 0.59ms


In [23]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="ornith-1.5:9b", base_url="http://localhost:11434", temperature=0.1)

In [14]:
result = llm.invoke("Quem é Sherlock Holmes?")
print(result)

content='**Sherlock Holmes** é um dos personagens fictícios mais famosos da história da literatura. Criado pelo escritor escocês **Sir Arthur Conan Doyle**, ele aparece pela primeira vez em *"Um Estudo em Escarlate"* (*A Study in Scarlet*), publicado em 1887.\n\n## Sobre o personagem\n\nSherlock Holmes é um **detetive britânico** famoso por sua genialidade, poder de observação e raciocínio lógico dedutivo. Ele vive em **221B Baker Street**, em Londres, no período vitoriano (final do século XIX).\n\nAlguns de seus traços marcantes:\n- **Observação apurada** — percebe detalhes que escapa aos outros\n- **Memória extraordinária**\n- Interesse por **química** e ciências\n- Toca violino\n- Comportamento às vezes considerado arrogante, excêntrico ou distante\n\n## A dupla Holmes e Watson\n\nO personagem é narrado e contado através de **Dr. John H. Watson**, seu companheiro e médico amigo. Juntos, investigam crimes e mistérios. Doyle se inspirou em um de seus amigos reais, o Dr. Douglas Fairba

In [7]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    base_url="http://localhost:11434", model="nomic-embed-text-v2-moe:latest"
)

In [8]:
result = embedding_model.embed_query("Quem é Sherlock Holmes?")
print(result)

[0.03046434, 0.0189809, -0.04618363, -0.00037145673, 0.029326903, -0.008694615, -0.050802574, 0.029355628, -0.015610213, 0.020645048, -0.0291308, -0.036489356, 0.05357741, 0.00084378786, 0.023622045, -0.08064386, 0.029090453, -0.0020195434, 0.025740772, 0.05036334, 0.057661526, 0.032921046, 0.03993349, -0.024216112, 0.002571287, 0.023780884, 0.0261563, -0.0128097795, 0.047095943, 0.032904033, -0.010681981, 0.036909796, -0.00051663053, 0.01190871, -0.0020830594, -0.00368897, -0.03932114, 0.026010728, 0.0097104795, 0.01518954, 0.010836549, 0.005902954, 0.015183526, 0.013944543, 0.017247925, 0.0035832387, -0.06550993, 0.03120733, 0.019850545, -0.012035124, 0.01994591, -0.0017588339, -0.031373497, -0.04605388, 0.038141448, -0.057281543, 0.05537421, -0.027998881, 0.02908977, 0.031673066, -0.01655057, 0.033299938, 0.09414602, -0.024796227, -0.0118426075, -0.03246028, -0.006041249, 0.018035829, -0.07435112, -0.02374129, -0.018628733, -0.01633585, -0.027638473, 0.045344085, -0.017316507, 0.025

# 1. Carregar dados e documentos

In [1]:
import json

with open("booklist.json", "r") as f:
    booklist = json.load(f)

In [53]:
!uv add langchain-community langchain-text-splitters

Resolved 92 packages in 1ms
Checked 85 packages in 0.37ms


In [2]:
from langchain_community.document_loaders import TextLoader
import os

documents = []
for book in booklist:
    loader = TextLoader(book["path"], encoding="utf-8")
    doc = loader.load()[0]
    doc.metadata["title"] = book["title"]
    doc.metadata["author"] = book["author"]
    doc.metadata["year"] = book["year"]
    doc.metadata["genre"] = book["genre"]
    doc.metadata["language"] = book["language"]
    documents.extend([doc])

print(f"Loaded {len(documents)} documents from {len(booklist)} books.")

/tmp/ipykernel_15909/1526419405.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 2 documents from 2 books.


In [3]:
print(documents[0].metadata)  # Print the metadata of the first document

{'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=250)
chunks = splitter.split_documents(documents)

In [5]:
print("Total number of chunks created:", len(chunks))

Total number of chunks created: 675


In [6]:
print(chunks[500])  # Print the metadata of the first chunk

page_content='“Forgive me, Doctor; I forgot myself. You do not need any help. I am so
worried in my mind that I am apt to be irritable. If you only knew the
problem I have to face, and that I am working out, you would pity, and
tolerate, and pardon me. Pray do not put me in a strait-waistcoat. I
want to think and I cannot think freely when my body is confined. I am
sure you will understand!” He had evidently self-control; so when the
attendants came I told them not to mind, and they withdrew. Renfield
watched them go; when the door was closed he said, with considerable
dignity and sweetness:--

“Dr. Seward, you have been very considerate towards me. Believe me that
I am very, very grateful to you!” I thought it well to leave him in this
mood, and so I came away. There is certainly something to ponder over in
this man’s state. Several points seem to make what the American
interviewer calls “a story,” if one could only get them in proper order.
Here they are:--

Will not mention “drinkin

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embedding_model)
vector_store.add_documents(chunks)

['b5c017d3-a57b-4ac2-a26c-c59a8e6f1db4',
 'd587352e-ce6c-488a-bf0a-1a64d364b731',
 'cc8c0ebb-67bb-4b87-bd56-5576ae150488',
 'feeb5ae2-672a-44a7-bbb9-ea697582682a',
 '089ad386-e481-4590-8476-9d46fb369694',
 '2abb7f1f-4f89-4bfd-b2c2-ccfe754b2d9d',
 '73d618cf-cbad-4d0e-83f7-be758531544a',
 '741e2bed-9555-4837-a623-1987961adcc2',
 'ee633288-7c30-459e-87fa-62d0db45973f',
 '57c958f2-f8b6-41a0-9762-afd80fc57834',
 'f8851696-8495-4444-b43a-1510d74b1f8c',
 '1003f5da-63d7-42cb-83b7-387eac9fbddd',
 '83031366-2409-458b-8ac0-00bb46269a67',
 '0fe55ae6-24ad-4688-aa04-be539e5c0e69',
 'eaafd75a-ca98-4309-ad09-d1726eb2042e',
 '1ff519f5-e3e5-4a84-905b-80d91d754e43',
 'ee2cfa37-8c4a-4ae6-a19c-59f8aa78af0a',
 '34c9ef88-da82-4444-ba66-1a18209848c1',
 'b0ec8387-df76-49e0-be2a-47736b3d421a',
 '5dba397d-7cdc-4257-a9d7-a6ab9acd4045',
 '2507df9f-979d-4d3d-8754-1850c5890b96',
 'b480fde8-0d9d-4f29-9cd2-93b7f344707b',
 '15cea068-1fc1-46e5-9949-d055730b60c7',
 '2bee2e59-f92d-408d-9488-1b445f7562a3',
 'ebc60c32-d3d5-

In [12]:
vector_store.similarity_search("What is Alice doing when she first notices the White Rabbit?", k=6)

[Document(id='ee633288-7c30-459e-87fa-62d0db45973f', metadata={'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}, page_content='Oh dear, what nonsense I’m talking!”\n\nJust then her head struck against the roof of the hall: in fact she was\nnow more than nine feet high, and she at once took up the little golden\nkey and hurried off to the garden door.\n\nPoor Alice! It was as much as she could do, lying down on one side, to\nlook through into the garden with one eye; but to get through was more\nhopeless than ever: she sat down and began to cry again.\n\n“You ought to be ashamed of yourself,” said Alice, “a great girl like\nyou,” (she might well say this), “to go on crying in this way! Stop\nthis moment, I tell you!” But she went on all the same, shedding\ngallons of tears, until there was a large pool all round her, about\nfour inches deep and reaching h

In [18]:
retriever = vector_store.as_retriever(search_kwargs={"k": 6}, search_type="similarity")
retriever.invoke("Who is Alice?")

[Document(id='fc4bdeae-9924-496d-a23f-1a30f9bfea91', metadata={'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}, page_content='First came ten soldiers carrying clubs; these were all shaped like the\nthree gardeners, oblong and flat, with their hands and feet at the\ncorners: next the ten courtiers; these were ornamented all over with\ndiamonds, and walked two and two, as the soldiers did. After these came\nthe royal children; there were ten of them, and the little dears came\njumping merrily along hand in hand, in couples: they were all\nornamented with hearts. Next came the guests, mostly Kings and Queens,\nand among them Alice recognised the White Rabbit: it was talking in a\nhurried nervous manner, smiling at everything that was said, and went\nby without noticing her. Then followed the Knave of Hearts, carrying\nthe King’s crown on a crimson velvet c

In [21]:
!uv add langchain-classic

3937.64s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 146 packages in 1.56s                                       ⠋ Resolving dependencies...                                                     
⠙ matplotlib-inline==0.2.2                                                      ⠋ Resolving dependencies...                                                     Checked 140 packages in 20ms


In [24]:
from langchain_classic.chains import RetrievalQA

chat = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True)

chat.invoke("Who is Alice?")

{'query': 'Who is Alice?',
 'result': 'Based on the context provided, Alice is the main character and narrator of this story (an excerpt from Lewis Carroll\'s *Alice\'s Adventures in Wonderland*).\n\nFrom what we can see in the text, Alice is:\n\n- **A curious young girl** — She\'s constantly observing and questioning her surroundings, from the strange procession of cards to the peculiar behavior of the White Rabbit.\n\n- **Thoughtful and self-reflective** — At one point she wonders aloud whether she\'s been changed into someone else, comparing herself to children she knows (like Ada and Mabel) and testing whether she still knows the things she used to know.\n\n- **Polite** — She addresses the Queen of Hearts respectfully, saying "please your Majesty."\n\n- **Emotionally vulnerable** — She cries frequently, at one point filling the hall with tears, and she scolds herself for crying.\n\n- **Determined** — Despite her frustrations (like being too big to fit through the garden door), she 

In [25]:
chat.invoke("Who is the White Rabbit?")

{'query': 'Who is the White Rabbit?',
 'result': 'The White Rabbit is a character in *Alice\'s Adventures in Wonderland* by Lewis Carroll. Based on the text you\'ve provided and the broader story, here\'s what he is:\n\n**His role:** The White Rabbit is a servant (essentially a footman or attendant) in the court of the Queen of Hearts. He is constantly anxious, late, and in a hurry, always checking his pocket watch and muttering to himself about being late.\n\n**Key details from the text:**\n- He is dressed formally, wearing a **waistcoat** and carrying a **pocket watch**, a **fan**, and **white kid gloves**.\n- He is so flustered that he addresses Alice as "Mary Ann" (his housemaid), mistakenly thinking she is his servant.\n- He is in a perpetual state of panic, worrying about the Duchess and the Queen ("Oh! the Duchess, the Duchess! Oh! won\'t she be savage if I\'ve kept her waiting!").\n- He is even mentioned as being "under sentence of execution" by another Rabbit.\n- He serves as 